In [48]:
import pandas as pd

In [49]:
ratings = pd.read_csv('/Users/sebastian/University/Master/third term/sem-proj/kg-token/data/ml-100k/u.data', header=None, sep='\t', encoding='latin1')
ratings.columns = ['user_id', 'movie_id', 'rating', 'timestamp']

In [50]:
ratings

,user_id,movie_id,rating,timestamp
0,196,242,3,881250949
1,186,302,3,891717742
2,22,377,1,878887116
3,244,51,2,880606923
4,166,346,1,886397596
...,...,...,...,...
99995,880,476,3,880175444
99996,716,204,5,879795543
99997,276,1090,1,874795795
99998,13,225,2,882399156


In [51]:
ratings['movie_id'] -= 1

In [55]:
ratings['user_id'] -= 1

In [57]:
ratings

,user_id,movie_id,rating,timestamp
0,195,241,3,881250949
1,185,301,3,891717742
2,21,376,1,878887116
3,243,50,2,880606923
4,165,345,1,886397596
...,...,...,...,...
99995,879,475,3,880175444
99996,715,203,5,879795543
99997,275,1089,1,874795795
99998,12,224,2,882399156


In [58]:
aggregated_df = ratings.groupby('user_id').agg(
    movies_rated=('movie_id', list),
    ratings=('rating', list)
).reset_index()

aggregated_df

,user_id,movies_rated,ratings
0,0,"[60, 188, 32, 159, 19, 201, 170, 264, 154, 116...","[4, 3, 4, 4, 4, 5, 5, 4, 2, 3, 4, 4, 5, 5, 4, ..."
1,1,"[291, 250, 49, 313, 296, 289, 311, 280, 12, 27...","[4, 5, 5, 1, 4, 3, 3, 3, 4, 3, 4, 3, 3, 4, 5, ..."
2,2,"[334, 244, 336, 342, 322, 330, 293, 331, 327, ...","[1, 1, 1, 3, 2, 4, 2, 1, 5, 3, 3, 1, 4, 2, 3, ..."
3,3,"[263, 302, 360, 356, 259, 355, 293, 287, 49, 3...","[3, 5, 5, 4, 4, 3, 5, 4, 5, 5, 4, 5, 3, 5, 3, ..."
4,4,"[1, 16, 438, 224, 109, 453, 423, 0, 362, 97, 1...","[3, 4, 1, 2, 1, 1, 1, 4, 3, 3, 3, 4, 5, 2, 4, ..."
...,...,...,...
938,938,"[930, 105, 257, 1053, 688, 475, 408, 120, 1189...","[2, 3, 4, 4, 5, 5, 4, 5, 5, 4, 5, 4, 5, 3, 2, ..."
939,939,"[192, 567, 13, 204, 271, 654, 314, 65, 872, 28...","[3, 3, 3, 3, 4, 4, 4, 4, 3, 3, 5, 5, 4, 3, 5, ..."
940,940,"[146, 123, 116, 180, 992, 257, 6, 474, 256, 14...","[4, 5, 5, 5, 4, 4, 4, 4, 4, 4, 4, 2, 2, 3, 5, ..."
941,941,"[116, 199, 603, 422, 260, 426, 486, 322, 614, ...","[4, 4, 4, 5, 4, 5, 4, 3, 3, 4, 5, 3, 3, 2, 4, ..."


In [59]:
def separate_movies_ratings(row):
    movies_high = [movie for movie, rating in zip(row['movies_rated'], row['ratings']) if rating > 3]
    movies_low = [movie for movie, rating in zip(row['movies_rated'], row['ratings']) if rating <= 3]
    return pd.Series([movies_high, movies_low])

aggregated_df[['movies_rated_gt_3', 'movies_rated_lte_3']] = aggregated_df.apply(separate_movies_ratings, axis=1)

aggregated_df = aggregated_df.drop(columns=['movies_rated', 'ratings'])

aggregated_df

,user_id,movies_rated_gt_3,movies_rated_lte_3
0,0,"[60, 32, 159, 19, 201, 170, 264, 46, 221, 252,...","[188, 154, 116, 16, 91, 265, 73, 30, 69, 26, 2..."
1,1,"[291, 250, 49, 296, 12, 302, 256, 315, 300, 31...","[313, 289, 311, 280, 279, 307, 306, 314, 297, ..."
2,2,"[330, 327, 317, 347, 326, 320, 259, 319, 341, ...","[334, 244, 336, 342, 322, 293, 331, 333, 349, ..."
3,3,"[302, 360, 356, 259, 293, 287, 49, 353, 270, 2...","[263, 355, 327, 209, 357]"
4,4,"[16, 0, 210, 381, 61, 23, 422, 266, 221, 172, ...","[1, 438, 224, 109, 453, 423, 362, 97, 101, 375..."
...,...,...,...
938,938,"[257, 1053, 688, 475, 408, 120, 1189, 992, 221...","[930, 105, 817, 889, 933, 265, 679, 423, 253, ..."
939,939,"[271, 654, 314, 65, 95, 193, 171, 7, 55, 146, ...","[192, 567, 13, 204, 872, 288, 180, 152, 354, 6..."
940,940,"[146, 123, 116, 180, 992, 257, 6, 474, 256, 14...","[221, 357, 762, 272]"
941,941,"[116, 199, 603, 422, 260, 426, 486, 583, 346, ...","[322, 614, 538, 312, 268, 361, 891, 321, 182, ..."


In [60]:
aggregated_df.to_csv('/Users/sebastian/University/Master/third term/sem-proj/kg-token/src/data/users_movies_full_split.csv', index=False)

In [41]:
aggregated_df['movies_rated_gt_3_count'] = aggregated_df['movies_rated_gt_3'].apply(len)
aggregated_df['movies_rated_lte_3_count'] = aggregated_df['movies_rated_lte_3'].apply(len)

min_gt_3 = aggregated_df['movies_rated_gt_3_count'].min()
max_gt_3 = aggregated_df['movies_rated_gt_3_count'].max()
min_lte_3 = aggregated_df['movies_rated_lte_3_count'].min()
max_lte_3 = aggregated_df['movies_rated_lte_3_count'].max()

min_gt_3, max_gt_3, min_lte_3, max_lte_3

(np.int64(0), np.int64(172), np.int64(0), np.int64(485))

In [42]:
aggregated_df

,user_id,movies_rated_gt_3,movies_rated_lte_3,movies_rated_gt_3_count,movies_rated_lte_3_count
0,1,"[201, 170, 252, 112, 63, 227, 113, 220, 59, 17...","[265, 73, 259, 139, 119, 103, 77, 142, 258, 11...",81,25
1,2,"[250, 49, 315, 312, 241, 282, 310, 99, 126, 28...","[313, 314, 293, 308]",13,4
2,3,"[327, 320, 319, 346, 339, 345]","[334, 244, 336, 331, 340, 324, 335, 352]",6,8
3,4,"[302, 360, 293, 49, 353, 299, 257, 328, 326, 3...",[],14,0
4,5,"[381, 435, 41, 152, 108, 99, 88, 432, 427, 208...","[438, 109, 453, 423, 376, 452, 240, 388, 410, ...",26,42
...,...,...,...,...,...
938,939,"[688, 475, 120, 1189, 221, 325, 1050, 1276, 21...",[],27,0
939,940,"[95, 193, 7, 55, 94, 854, 426, 299, 481, 708, ...","[354, 609, 1400, 357, 263]",14,5
940,941,"[123, 116, 180, 297, 407, 918, 0]",[],7,0
941,942,"[422, 426, 346, 303, 192, 30, 130, 499, 171, 6...",[],35,0


In [43]:
# Filter users with less than 5
filtered_df = aggregated_df[
    (aggregated_df['movies_rated_gt_3_count'] >= 5) & 
    (aggregated_df['movies_rated_lte_3_count'] >= 5)
].reset_index(drop=True)

filtered_df

,user_id,movies_rated_gt_3,movies_rated_lte_3,movies_rated_gt_3_count,movies_rated_lte_3_count
0,1,"[201, 170, 252, 112, 63, 227, 113, 220, 59, 17...","[265, 73, 259, 139, 119, 103, 77, 142, 258, 11...",81,25
1,3,"[327, 320, 319, 346, 339, 345]","[334, 244, 336, 331, 340, 324, 335, 352]",6,8
2,5,"[381, 435, 41, 152, 108, 99, 88, 432, 427, 208...","[438, 109, 453, 423, 376, 452, 240, 388, 410, ...",26,42
3,6,"[13, 97, 491, 468, 210, 474, 133, 524, 522, 48...","[476, 475, 457, 464, 258, 404, 471]",41,7
4,7,"[491, 660, 647, 377, 199, 643, 450, 80, 575, 5...","[566, 668, 598, 259, 211, 680, 218, 293, 323, ...",161,14
...,...,...,...,...,...
280,933,"[63, 97, 178, 473, 126, 55, 179, 99, 21]","[576, 230, 432, 218, 549, 109, 933, 450, 1245,...",9,42
281,934,"[49, 196, 296, 201, 222, 404, 207, 1448, 173, ...","[1424, 1036, 817, 208, 1310, 785, 215]",29,7
282,938,"[312, 1011, 180, 369, 1027, 110, 126, 865, 49,...","[121, 476, 104, 828, 254, 234, 1253, 455, 409,...",21,12
283,940,"[95, 193, 7, 55, 94, 854, 426, 299, 481, 708, ...","[354, 609, 1400, 357, 263]",14,5


In [44]:
import numpy as np
def select_random_movies(row, n=5):
    movies_gt_3 = np.random.choice(row['movies_rated_gt_3'], size=min(len(row['movies_rated_gt_3']), n), replace=False).tolist()
    movies_lte_3 = np.random.choice(row['movies_rated_lte_3'], size=min(len(row['movies_rated_lte_3']), n), replace=False).tolist()
    return pd.Series([movies_gt_3, movies_lte_3])

filtered_df[['random_movies_gt_3', 'random_movies_lte_3']] = filtered_df.apply(select_random_movies, axis=1)

filtered_df = filtered_df.drop(columns=['movies_rated_gt_3_count', 'movies_rated_lte_3_count'])

filtered_df


,user_id,movies_rated_gt_3,movies_rated_lte_3,random_movies_gt_3,random_movies_lte_3
0,1,"[201, 170, 252, 112, 63, 227, 113, 220, 59, 17...","[265, 73, 259, 139, 119, 103, 77, 142, 258, 11...","[151, 267, 86, 108, 112]","[259, 253, 103, 7, 28]"
1,3,"[327, 320, 319, 346, 339, 345]","[334, 244, 336, 331, 340, 324, 335, 352]","[339, 319, 346, 320, 345]","[334, 340, 335, 331, 336]"
2,5,"[381, 435, 41, 152, 108, 99, 88, 432, 427, 208...","[438, 109, 453, 423, 376, 452, 240, 388, 410, ...","[100, 433, 152, 99, 208]","[453, 388, 215, 376, 368]"
3,6,"[13, 97, 491, 468, 210, 474, 133, 524, 522, 48...","[476, 475, 457, 464, 258, 404, 471]","[492, 126, 489, 483, 484]","[258, 476, 404, 457, 471]"
4,7,"[491, 660, 647, 377, 199, 643, 450, 80, 575, 5...","[566, 668, 598, 259, 211, 680, 218, 293, 323, ...","[306, 633, 501, 153, 631]","[323, 598, 259, 439, 144]"
...,...,...,...,...,...
280,933,"[63, 97, 178, 473, 126, 55, 179, 99, 21]","[576, 230, 432, 218, 549, 109, 933, 450, 1245,...","[21, 473, 126, 97, 179]","[448, 452, 109, 52, 678]"
281,934,"[49, 196, 296, 201, 222, 404, 207, 1448, 173, ...","[1424, 1036, 817, 208, 1310, 785, 215]","[88, 173, 68, 413, 196]","[1310, 817, 215, 1036, 208]"
282,938,"[312, 1011, 180, 369, 1027, 110, 126, 865, 49,...","[121, 476, 104, 828, 254, 234, 1253, 455, 409,...","[120, 927, 257, 180, 272]","[844, 476, 121, 409, 234]"
283,940,"[95, 193, 7, 55, 94, 854, 426, 299, 481, 708, ...","[354, 609, 1400, 357, 263]","[299, 854, 257, 7, 426]","[357, 1400, 263, 609, 354]"


In [47]:
filtered_df.to_csv('/Users/sebastian/University/Master/third term/sem-proj/kg-token/src/data/users_movies_15_split.csv', index=False)